# Positional Encoding + Self-attention

In [1]:
import tensorflow as tf
import numpy as np

from tensorflow.keras.layers import TextVectorization, Embedding, MultiHeadAttention

## Input Sequence

In [2]:
sentence = ['I love deep learning']

In [4]:
vectorizer = TextVectorization(output_mode='int')
vectorizer.adapt(sentence)

tokens = vectorizer(sentence)
print('Vocabulary: ', vectorizer.get_vocabulary())

print('Tokens: ', tokens.numpy())

Vocabulary:  ['', '[UNK]', np.str_('love'), np.str_('learning'), np.str_('i'), np.str_('deep')]
Tokens:  [[4 2 5 3]]


In [6]:
# word embeddings
embedding_dim = 8
embedding_layer = Embedding(input_dim=vectorizer.vocabulary_size(), output_dim=embedding_dim)
word_embeddings=embedding_layer(tokens)
word_embeddings

<tf.Tensor: shape=(1, 4, 8), dtype=float32, numpy=
array([[[ 0.04883465,  0.00746737, -0.03166398,  0.00291563,
         -0.04483116,  0.02410522,  0.01698551, -0.02455443],
        [ 0.02083205,  0.00506991, -0.0054345 ,  0.0399019 ,
         -0.01216208,  0.00602566, -0.00163809, -0.04351738],
        [-0.049233  , -0.01423382,  0.02014157,  0.01855887,
         -0.01102393, -0.01067172, -0.03823296, -0.02687529],
        [ 0.02055954, -0.00277716, -0.0043662 ,  0.00523828,
         -0.01109225,  0.03974624, -0.04475931, -0.01644281]]],
      dtype=float32)>

In [7]:
# Positional encoding function
def positional_encoding(max_position, d_model):
  positions = np.arange(max_position)[:, np.newaxis]
  dimensions = np.arange(d_model)[np.newaxis, :]
  angle_rates = 1 / np.power(10000, (2 * (dimensions // 2)) / np.float32(d_model))
  angle_rads = positions * angle_rates
  pos_encoding = np.concatenate([np.sin(angle_rads), np.cos(angle_rads)], axis=-1)
  return tf.cast(pos_encoding, dtype=tf.float32)

In [8]:
positional_encoding(4, embedding_dim)

<tf.Tensor: shape=(4, 16), dtype=float32, numpy=
array([[ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00,
         0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00,
         1.0000000e+00,  1.0000000e+00,  1.0000000e+00,  1.0000000e+00,
         1.0000000e+00,  1.0000000e+00,  1.0000000e+00,  1.0000000e+00],
       [ 8.4147096e-01,  8.4147096e-01,  9.9833414e-02,  9.9833414e-02,
         9.9998331e-03,  9.9998331e-03,  9.9999981e-04,  9.9999981e-04,
         5.4030228e-01,  5.4030228e-01,  9.9500418e-01,  9.9500418e-01,
         9.9994999e-01,  9.9994999e-01,  9.9999952e-01,  9.9999952e-01],
       [ 9.0929741e-01,  9.0929741e-01,  1.9866933e-01,  1.9866933e-01,
         1.9998666e-02,  1.9998666e-02,  1.9999987e-03,  1.9999987e-03,
        -4.1614684e-01, -4.1614684e-01,  9.8006660e-01,  9.8006660e-01,
         9.9980003e-01,  9.9980003e-01,  9.9999797e-01,  9.9999797e-01],
       [ 1.4112000e-01,  1.4112000e-01,  2.9552022e-01,  2.9552022e-01,
         2.9

In [9]:
# Multi-head attention
attention_layer = MultiHeadAttention(
    num_heads=2,
    key_dim=embedding_dim
)

In [11]:
# Apply self attention
attention_output = attention_layer(
    query=word_embeddings,
    value=word_embeddings,
    key=word_embeddings
)

attention_output.shape

TensorShape([1, 4, 8])